# Silver Layer: Product Dimension
**Objective:** Cleanse, standardize, and deduplicate the Product Dimension table for Gold Layer consumption.

## Key Transformations
* **Deduplication:** Applied Key-Level deduplication (`dropDuplicates(['product_key'])`) to resolve  duplicate records and prevent Fan-Out traps in Fact tables.
* **Financial Standardization:** Extracted `currency` (USD, EUR, GBP) and cast `standard_cost` to `Decimal(18,2)`.
* **String Cleansing:** Capitalized `color` values consistently (e.g., 'MULTI' -> 'Multi') and replaced implicit nulls with `'Unknown'`.

In [0]:
from pyspark.sql import functions as F

# --- Step 1: Load Bronze Table ---
df = spark.table("bronze.bronze_product")

# --- Step 2: Universal String Cleaning ---
# Trim, remove '|', and map pseudo-nulls to actual NULLs
string_cols = [c for c, t in df.dtypes if t == "string"]
for c in string_cols:
    cleaned = F.trim(F.regexp_replace(F.col(c), r"\|", ""))
    df = df.withColumn(c, 
        F.when(F.upper(cleaned).isin("N/A", "NA", "NULL", "NONE", "-", ""), None)
         .otherwise(cleaned)
    )

# --- Step 3: Standardize Colors & Impute Nulls ---
if "color" in df.columns:
    # Capitalize first letter consistently (e.g., 'Multi', 'Red')
    df = df.withColumn("color", F.initcap(F.lower(F.col("color"))))
    # Replace remaining Nulls with 'Unknown' for clean BI reporting
    df = df.fillna("Unknown", subset=["color"])

# --- Step 4: Split Standard Cost & Currency Detection ---
if "standard_cost" in df.columns:
    # Detect Currency (Using raw string 'r' to prevent SyntaxWarnings)
    df = df.withColumn("currency",
        F.when(F.col("standard_cost").rlike(r"\$"), "USD")
         .when(F.col("standard_cost").rlike("€"), "EUR")
         .when(F.col("standard_cost").rlike("£"), "GBP")
         .otherwise("USD") # Defaulting to USD if no symbol is found
    )
    
    # Clean the string and cast directly to Decimal(18,2) for financial accuracy
    df = df.withColumn("standard_cost_value", 
        F.regexp_replace(F.col("standard_cost"), r"[$,€£\s]", "").cast("decimal(18,2)")
    ).drop("standard_cost")

# --- Step 5: Optimization & Deduplication (Prevent Fan-Out Trap) ---
# Ensure Keys are Integers
df = df.withColumn("product_key", F.col("product_key").cast("int"))

# Remove null keys and duplicate IDs (Crucial for Dimension Tables)
df = df.filter(F.col("product_key").isNotNull())
df = df.dropDuplicates(["product_key"])

# --- Step 6: Metadata & Ordering ---
df = df.withColumn("silver_processed_at", F.current_timestamp())

# Ensure ingestion_timestamp exists if not passed from Bronze
if "ingestion_timestamp" not in df.columns:
    df = df.withColumn("ingestion_timestamp", F.current_timestamp())

# Reorder logically for the data warehouse
preferred_order = ["product_key", "product", "standard_cost_value", "currency", "color"]
existing_preferred = [c for c in preferred_order if c in df.columns]
rest = [c for c in df.columns if c not in existing_preferred and c not in ["ingestion_timestamp", "silver_processed_at"]]

df = df.select(*existing_preferred, *rest, "ingestion_timestamp", "silver_processed_at")

# --- Step 7: Save to Silver ---
df.write.mode("overwrite") \
    .option("overwriteSchema", "true") \
    .format("delta") \
    .saveAsTable("silver.silver_product")

print("Pipeline Status: Success. Silver Table 'silver_product' updated.")
display(df.orderBy("product_key").limit(100))